# SOB4ES - XGBoost Multisalida
### CRISP-ML(Q) - Fase 3 y 4: Desarrollo y evaluacion de los modelos

Este notebook implementa **XGBoost con salida vectorial**, disponible desde la version 1.7 mediante el parametro `multi_strategy='multi_output_tree'`.

A diferencia del enfoque individual (un modelo por target), aqui un unico arbol de decision en cada iteracion del boosting aprende a predecir los 11 targets simultaneamente. Esto permite capturar correlaciones entre grupos biologicos de forma mas directa que entrenar modelos separados.

**Datasets utilizados:**
- `train.csv` - Entrenamiento (70 %, estratificado por pais)
- `test.csv`  - Tuning e hiperparametros (15 %)
- `eval.csv`  - Evaluacion final imparcial (15 %)

**Estructura del notebook:**
1. Configuracion
2. Carga de datos
3. Tuning de hiperparametros
4. Validacion cruzada
5. Importancia de variables
6. Entrenamiento de produccion
7. Evaluacion final

---
## 1. Configuracion del entorno


In [8]:
import os
import time
import warnings
import joblib
import pandas as pd
import numpy as np
from pathlib import Path

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_validate, RepeatedKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')

DATA_DIR   = 'input/'
MODELS_DIR = 'output/models/xgb_multi/'
Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)

TARGETS = [
    # Shannon diversity index
    'nematode_shannon_z',
    'macro_shannon_z',
    'earthworm_shannon_z',
    'orib_shannon_z',
    'meso_shannon_z',
    'coll_shannon_z',
    'bac_shannon_z',
    'fun_shannon_z',
    'euk_shannon_z',
    'oomy_shannon_z',
    'cerc_shannon_z',
    # Richness
    'macro_order_richness_z',
    'earthworm_richness_z',
    'orib_species_richness_z',
    'meso_species_richness_z',
    'coll_species_richness_z',
    'bac_asv_richness_z',
    'fun_asv_richness_z',
    'euk_asv_richness_z',
    'oomy_asv_richness_z',
    'cerc_asv_richness_z'
]


FEATURES_AUTORIZADAS = [
    'total_plant_cover_z',
    'clay_content_z',
    'silt_content_z',
    'sand_content_z',
    'aggregate_stability_z',
    'bulk_density_z',
    'soil_moisture_z',
    'as_z',
    'cu_z',
    'k_z',
    'mo_z',
    'ni_z',
    'p_z',
    'pb_z',
    'zn_z',
    'soil_ph_z',
    'plot_total_organic_c_z',
    'plot_total_n_z',
    'gee_temp_media_C_z',
    'gee_humedad_rel_pct_z',
    'gee_ndvi_verano_z',
    'dem_elevacion_m_z',
    'dem_pendiente_deg_z',
    'dem_orientacion_deg_z',
    'eu_clay_content_z',
    'eu_sand_content_z',
    'eu_silt_content_z',
    'eu_water_holding_capacity_z',
    'eu_cn_ratio_z',
    'eu_p_z',
    'eu_ph_z',
    'eu_as_z',
    'eu_organic_carbon_octop_z',
    'eu_zn_z'
]


print('Librerias y variables cargadas.')


Librerias y variables cargadas.


## 2. Carga de datos

Se cargan los tres datasets pre-generados (division 70/15/15, estratificada por pais).

| Archivo    | Uso                                      | Filas aprox. |
|------------|------------------------------------------|--------------|
| `train.csv`| Ajuste del modelo                        | 299          |
| `test.csv` | Tuning / validacion cruzada              | 64           |
| `eval.csv` | Evaluacion final imparcial               | 65           |

El dataset `eval.csv` no se utiliza hasta la seccion 7.

In [9]:
def cargar_csv(ruta, nombre):
    if not os.path.exists(ruta):
        raise FileNotFoundError(f'No se encontro: {ruta}')
    try:
        df = pd.read_csv(ruta, sep=',')
        if len(df.columns) < 5:
            df = pd.read_csv(ruta, sep=';')
    except Exception:
        df = pd.read_csv(ruta, sep=';')
    print(f'  {nombre}: {df.shape[0]} filas x {df.shape[1]} columnas')
    return df

print('Cargando datasets...')
df_train = cargar_csv(os.path.join(DATA_DIR, 'train.csv'), 'train.csv')
df_test  = cargar_csv(os.path.join(DATA_DIR, 'test.csv'),  'test.csv')
df_eval  = cargar_csv(os.path.join(DATA_DIR, 'eval.csv'),  'eval.csv')

X_train = df_train[FEATURES_AUTORIZADAS]
X_test  = df_test[FEATURES_AUTORIZADAS]
X_eval  = df_eval[FEATURES_AUTORIZADAS]

# En los modelos multisalida, y es una matriz 2D (n_muestras x n_targets)
# en lugar de un vector 1D. Esto permite al modelo aprender correlaciones entre targets.
y_train = df_train[TARGETS]
y_test  = df_test[TARGETS]
y_eval  = df_eval[TARGETS]

print(f'\nDatasets cargados.')
print(f'X_train: {X_train.shape}  |  y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}   |  y_test:  {y_test.shape}')
print(f'X_eval:  {X_eval.shape}   |  y_eval:  {y_eval.shape}')


Cargando datasets...
  train.csv: 299 filas x 71 columnas
  test.csv: 64 filas x 71 columnas
  eval.csv: 65 filas x 71 columnas

Datasets cargados.
X_train: (299, 44)  |  y_train: (299, 11)
X_test:  (64, 44)   |  y_test:  (64, 11)
X_eval:  (65, 44)   |  y_eval:  (65, 11)


## 3. Busqueda de hiperparametros (Tuning)

El espacio de busqueda es identico al del XGBoost individual. El parametro `multi_strategy='multi_output_tree'` activa la salida vectorial: en lugar de construir un arbol por target (comportamiento por defecto), construye un unico arbol que predice todos los targets a la vez en cada iteracion del boosting.

Al igual que con RF multisalida, el tuning usa un unico target de referencia para el scoring y los parametros optimos se aplican despues al modelo completo multisalida.

In [10]:
print('Iniciando busqueda de hiperparametros...')
print('Este proceso puede tardar varios minutos.\n')

param_grid = {
    'n_estimators':      [100, 300, 500, 800, 1000],
    'max_depth':         [3, 4, 5, 6, 7],
    'learning_rate':     [0.01, 0.05, 0.1, 0.15],
    'subsample':         [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree':  [0.6, 0.7, 0.8, 0.9, 1.0],
    'gamma':             [0, 0.1, 0.2, 0.5, 1.0],
    'min_child_weight':  [1, 3, 5, 7]
}

y_ref = y_train['earthworm_shannon_z'].values

buscador = RandomizedSearchCV(
    estimator=XGBRegressor(tree_method='hist', random_state=42),
    param_distributions=param_grid,
    n_iter=50, scoring='r2', cv=5,
    verbose=1, random_state=42, n_jobs=-1
)

start = time.time()
buscador.fit(X_train, y_ref)

print(f'\nBusqueda completada en {(time.time()-start)/60:.1f} minutos.')
print(f'Mejor R2 de referencia: {buscador.best_score_:.4f}')
print(f'Hiperparametros optimos: {buscador.best_params_}')

XGB_MULTI_PARAMS = buscador.best_params_
XGB_MULTI_PARAMS['tree_method']      = 'hist'
XGB_MULTI_PARAMS['multi_strategy']   = 'multi_output_tree'
XGB_MULTI_PARAMS['random_state']     = 42


Iniciando busqueda de hiperparametros...
Este proceso puede tardar varios minutos.

Fitting 5 folds for each of 50 candidates, totalling 250 fits



Busqueda completada en 1.1 minutos.
Mejor R2 de referencia: 0.4949
Hiperparametros optimos: {'subsample': 0.8, 'n_estimators': 800, 'min_child_weight': 7, 'max_depth': 5, 'learning_rate': 0.05, 'gamma': 1.0, 'colsample_bytree': 0.6}


## 4. Validacion cruzada

Se evalua la robustez con validacion cruzada repetida (5 pliegues x 3 repeticiones) sobre `X_train` con todos los targets a la vez. Se muestra el R2 promedio global y el R2 individual por target.

In [11]:
print('Validacion cruzada repetida sobre X_train (todos los targets)...\n')

modelo_cv   = XGBRegressor(**XGB_MULTI_PARAMS)
cv_splitter = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

resultados = cross_validate(
    estimator=modelo_cv,
    X=X_train, y=y_train,
    cv=cv_splitter,
    scoring='r2',
    return_train_score=True
)

r2_train = np.mean(resultados['train_score'])
r2_cv    = np.mean(resultados['test_score'])
r2_std   = np.std(resultados['test_score'])

print(f'  Train R2 (promedio targets): {r2_train:.4f}')
print(f'  Val   R2 (promedio targets): {r2_cv:.4f} +/- {r2_std:.4f}')

# R2 individual por target
modelo_cv.fit(X_train, y_train)
y_pred_train = modelo_cv.predict(X_train)
print('\n  R2 por target (train):')
for i, t in enumerate(TARGETS):
    r2 = r2_score(y_train.iloc[:, i], y_pred_train[:, i])
    print(f'    {t:30} {r2:.4f}')


Validacion cruzada repetida sobre X_train (todos los targets)...

  Train R2 (promedio targets): 0.9447
  Val   R2 (promedio targets): 0.3150 +/- 0.0274

  R2 por target (train):
    nematode_shannon_z             0.9388
    macro_shannon_z                0.9436
    earthworm_shannon_z            0.9519
    orib_shannon_z                 0.9235
    meso_shannon_z                 0.9285
    coll_shannon_z                 0.9443
    bac_shannon_z                  0.9718
    fun_shannon_z                  0.9277
    euk_shannon_z                  0.9517
    oomy_shannon_z                 0.9489
    cerc_shannon_z                 0.9540


## 5. Importancia de variables

XGBoost expone directamente la importancia de cada variable mediante `feature_importances_`, que en modo multisalida refleja el impacto global sobre todos los targets. Se complementa con permutation importance para una estimacion mas fiable.

In [12]:
print('Calculando importancia de variables...\n')

# Importancia intrinseca de XGBoost (gain medio por variable)
imp_xgb = pd.Series(modelo_cv.feature_importances_, index=FEATURES_AUTORIZADAS)

# Permutation importance sobre X_train
perm = permutation_importance(
    modelo_cv, X_train, y_train,
    n_repeats=10, random_state=42, n_jobs=-1
)
imp_perm = pd.Series(perm.importances_mean, index=FEATURES_AUTORIZADAS)

top10 = pd.DataFrame({
    'XGBoost gain':      imp_xgb,
    'Permutacion (R2)':  imp_perm
}).sort_values('Permutacion (R2)', ascending=False).head(10).round(4)

print('Top 10 variables por importancia global (todos los targets):')
print(top10.to_string())


Calculando importancia de variables...

Top 10 variables por importancia global (todos los targets):
                     XGBoost gain  Permutacion (R2)
bac_asv_richness_z         0.0571            0.1838
euk_asv_richness_z         0.0413            0.1303
cerc_asv_richness_z        0.0304            0.0705
oomy_asv_richness_z        0.0254            0.0603
cerc_total_reads_z         0.0239            0.0602
oomy_total_reads_z         0.0165            0.0413
soil_ph_z                  0.0318            0.0336
euk_total_reads_z          0.0197            0.0317
gee_temp_media_C_z         0.0545            0.0307
eu_as_z                    0.0317            0.0257


## 6. Entrenamiento de produccion

Se entrena un ensamble de 5 modelos XGBoost multisalida, variando la semilla aleatoria. Cada modelo predice los 11 targets a la vez con `multi_strategy='multi_output_tree'`.

In [13]:
print('Entrenamiento de produccion (ensamble de 5 modelos XGBoost multisalida)...')
print('-' * 70)

global_start = time.time()
NUM_EXPERTOS = 5

for exp_id in range(NUM_EXPERTOS):
    t0     = time.time()
    params = XGB_MULTI_PARAMS.copy()
    params['random_state'] = 42 + exp_id

    # Entrenamos sobre y_train completo (matriz 11 targets)
    model  = XGBRegressor(**params)
    model.fit(X_train, y_train)

    joblib.dump(model, os.path.join(MODELS_DIR, f'xgb_multi_exp{exp_id}.pkl'))
    print(f'  Modelo {exp_id+1}/5 completado ({time.time()-t0:.1f}s)')

print('-' * 70)
print(f'Produccion completada en {(time.time()-global_start)/60:.1f} minutos.')


Entrenamiento de produccion (ensamble de 5 modelos XGBoost multisalida)...
----------------------------------------------------------------------
  Modelo 1/5 completado (18.6s)
  Modelo 2/5 completado (16.2s)
  Modelo 3/5 completado (18.2s)
  Modelo 4/5 completado (17.1s)
  Modelo 5/5 completado (18.5s)
----------------------------------------------------------------------
Produccion completada en 1.5 minutos.


## 7. Evaluacion final y prueba de funcionamiento

Esta seccion tiene dos partes:

**Parte A - Evaluacion final**
Se mide por primera vez el rendimiento sobre `eval.csv`, el conjunto de datos que el modelo nunca ha visto. Al ser un modelo multisalida, se calculan las metricas globales (promedio sobre todos los targets) y tambien el R2 individual por target, para identificar cuales predice bien y cuales no.

**Parte B - Prueba de funcionamiento (smoke test)**
Se verificia que el pipeline funciona de extremo a extremo con 3 filas reales de `eval.csv`, comparando la prediccion con el valor real observado para cada target.

---
**Como leer el indice Shannon H':**
Cada target es el indice de diversidad de Shannon de un grupo biologico distinto (lombrices, colémbolos, bacterias, hongos, etc.). El valor va de `0.0` (sin diversidad) a `4.0-5.0` (alta diversidad). Un valor tipico en suelos agricolas europeos se situa entre `1.5` y `2.5`.

In [14]:
NUM_EXPERTOS = 5

# ─────────────────────────────────────────────────────────────────────────
# PARTE A: Evaluacion final
# ─────────────────────────────────────────────────────────────────────────
print('Evaluacion final sobre eval.csv')
print('-' * 60)

preds_eval = []
for exp_id in range(NUM_EXPERTOS):
    ruta  = os.path.join(MODELS_DIR, f'xgb_multi_exp{exp_id}.pkl')
    model = joblib.load(ruta)
    preds_eval.append(model.predict(X_eval))

y_pred_eval = np.mean(preds_eval, axis=0)
y_true_eval = y_eval.values

r2_global   = r2_score(y_true_eval, y_pred_eval, multioutput='uniform_average')
rmse_global = np.sqrt(mean_squared_error(y_true_eval, y_pred_eval, multioutput='uniform_average'))
mae_global  = mean_absolute_error(y_true_eval, y_pred_eval, multioutput='uniform_average')

print(f'  R2   global: {r2_global:.4f}')
print(f'  RMSE global: {rmse_global:.4f} puntos de Shannon H\'')
print(f'  MAE  global: {mae_global:.4f} puntos de Shannon H\'')

print('\n  R2 por target:')
r2_por_target = r2_score(y_true_eval, y_pred_eval, multioutput='raw_values')
for t, r2 in zip(TARGETS, r2_por_target):
    print(f'    {t:30} {r2:.4f}')

# ─────────────────────────────────────────────────────────────────────────
# PARTE B: Smoke test con datos reales
# ─────────────────────────────────────────────────────────────────────────
print('\nSmoke test - Inferencia con datos reales')
print('-' * 60)

try:
    df_new_data      = X_eval.sample(n=3, random_state=42)
    indices_elegidos = df_new_data.index
    valores_reales   = y_eval.loc[indices_elegidos].values

    votos = []
    for exp_id in range(NUM_EXPERTOS):
        ruta  = os.path.join(MODELS_DIR, f'xgb_multi_exp{exp_id}.pkl')
        m     = joblib.load(ruta)
        pred  = m.predict(df_new_data)
        votos.append(pred)
        # pred es una matriz (3 muestras x 11 targets)
        print(f'  Modelo {exp_id+1} | '
              f'Fila {indices_elegidos[0]}: {pred[0, 0]:.4f} | '
              f'Fila {indices_elegidos[1]}: {pred[1, 0]:.4f} | '
              f'Fila {indices_elegidos[2]}: {pred[2, 0]:.4f}  (earthworm_shannon_z)')

    consenso = np.mean(votos, axis=0)
    print('-' * 60)
    print('  Resultados detallados por muestra (earthworm_shannon_z):')
    print('-' * 60)

    for i, idx in enumerate(indices_elegidos):
        pred_val = consenso[i, TARGETS.index('earthworm_shannon_z')]
        real_val = y_eval.loc[idx, 'earthworm_shannon_z']
        error    = abs(pred_val - real_val)
        print(f'    Fila {idx:<4} | Prediccion: {pred_val:.4f} | '
              f'Valor Real: {real_val:.4f} | Error absoluto: {error:.4f}')

    print('-' * 60)
    print('\nSmoke test superado. El pipeline XGBoost multisalida funciona correctamente.')

except Exception as e:
    print(f'Error en el smoke test: {str(e)}')


Evaluacion final sobre eval.csv
------------------------------------------------------------
  R2   global: 0.2685
  RMSE global: 0.8364 puntos de Shannon H'
  MAE  global: 0.6215 puntos de Shannon H'

  R2 por target:
    nematode_shannon_z             0.4654
    macro_shannon_z                0.3391
    earthworm_shannon_z            0.4779
    orib_shannon_z                 0.1944
    meso_shannon_z                 -0.3768
    coll_shannon_z                 -0.1967
    bac_shannon_z                  0.8240
    fun_shannon_z                  0.0426
    euk_shannon_z                  0.5435
    oomy_shannon_z                 0.3470
    cerc_shannon_z                 0.2933

Smoke test - Inferencia con datos reales
------------------------------------------------------------
  Modelo 1 | Fila 53: 0.1919 | Fila 60: -0.0759 | Fila 0: -0.5413  (earthworm_shannon_z)
  Modelo 2 | Fila 53: 0.4890 | Fila 60: 0.0566 | Fila 0: -0.7172  (earthworm_shannon_z)
  Modelo 3 | Fila 53: 0.2340 | Fila 6